In [2]:
import torch
from torch import nn

In [3]:
def corr2d(x,k):
    height,width = k.shape
    h = x.shape[0]-height+1
    w = x.shape[1]-width+1
    y = x.new_zeros(h,w)
    for i in range(h):
        for j in range(w):
            y[i,j] = (x[i:i+height,j:j+width]*k).sum()
            
    return y
x = torch.tensor([
    [0., 1., 2.],
    [3., 4., 5.],
    [6., 7., 8.]
])

k = torch.tensor([
    [0., 1.],
    [2., 3.]
])

y = corr2d(x,k)
print(y)


tensor([[19., 25.],
        [37., 43.]])


In [4]:
class conv2d(nn.Module):
    def __init__(self,k):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(*k))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self,x):
        return corr2d(x,self.weight)+self.bias
x = torch.randn(5,7)
k = torch.randn(2,3)        
net = conv2d((2,3))
y = net(x)
print(y)
print(y.shape)


tensor([[ 2.0862, -0.0700, -3.3842,  2.2995,  1.4812],
        [ 1.8333, -2.2393, -1.4064,  4.3022,  1.0467],
        [-2.3985, -0.5303,  1.0074,  0.4779, -3.1525],
        [-1.2298, -1.6685, -0.2758,  1.1424,  0.5659]], grad_fn=<AddBackward0>)
torch.Size([4, 5])


In [5]:
net = conv2d((2, 3))
x = torch.ones(5, 7)
y = net(x)
loss = y.sum()
loss.backward()
print(net.weight.grad.shape)
print(net.bias.grad)

torch.Size([2, 3])
tensor([20.])


In [6]:
x = torch.ones(6, 8)
x[:, 2:6] = 0

print(x)

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])


In [7]:
k = torch.tensor([[1,-1]])
y = corr2d(x,k)
y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [8]:
import torch
from torch import nn

torch.manual_seed(42)

# 数据
X = torch.ones(6, 8)
X[:, 2:6] = 0

# 用已知的卷积核构造目标结果
K = torch.tensor([[1., -1.]])
Y = corr2d(X, K)

# Conv2d 要求四维输入 (N, C, H, W)
X = X.reshape(1, 1, 6, 8)
Y = Y.reshape(1, 1, 6, 7)

# 创建需要训练的卷积层
net = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=(1, 2),
    bias=False
)

optimizer = torch.optim.SGD(
    net.parameters(), lr=0.03
)

print("初始权重:", net.weight.data)

# 开始训练
for epoch in range(30):
    optimizer.zero_grad()

    Y_hat = net(X)
    loss = ((Y_hat - Y) ** 2).sum()

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(
            epoch + 1,
            loss.item(),
            net.weight.data
        )

初始权重: tensor([[[[0.5406, 0.5869]]]])
5 3.5528934001922607 tensor([[[[ 0.7054, -1.0749]]]])
10 0.34759947657585144 tensor([[[[ 1.0487, -0.9277]]]])
15 0.036932382732629776 tensor([[[[ 0.9789, -1.0186]]]])
20 0.003961050417274237 tensor([[[[ 1.0064, -0.9934]]]])
25 0.0004252580110915005 tensor([[[[ 0.9979, -1.0021]]]])
30 4.565597191685811e-05 tensor([[[[ 1.0007, -0.9993]]]])
